# exp07 — Prompt Attention Analysis

**Question:** Does elevated attention in stego/control conditions reflect attention *to the prompt* (where letter constraints are specified), rather than attention *between sentence starts* within the CoT?

**Context:** exp06-control showed control elevation ≈ 95.8% of stego baseline — the effect is structural (letter constraint in prompt), not specific to steganographic hiding. Colleague hypothesis: model looks *back to the prompt* for the next letter, not at previous sentence starts.

**Two metrics measured in parallel:**

| Metric | What it measures |
|---|---|
| elevation | attn[pos_K → prev sent starts within CoT], relative to open |
| prompt_attn | attn[pos_K → prompt tokens], absolute value per condition |

**Three conditions:**

| Condition | Prompt has letter spec? | Source |
|---|---|---|
| open | No | results/exp01/valid_pairs.json |
| control | Yes (openly) | results/exp06_control/control_data.json |
| stego | Yes (hidden) | results/exp01/valid_pairs.json |

**Hypothesis (colleague Tanselu):** `prompt_attn: control ≈ stego >> open` — both constrained conditions attend back to the prompt for letter guidance.

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata, drive
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))

    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate scipy')

    # control_data.json was generated in exp06 and saved only to Drive
    drive.mount('/content/drive')
    import shutil
    os.makedirs('results/exp06_control', exist_ok=True)
    shutil.copy(
        '/content/drive/MyDrive/stego_cot_results/exp06_control/control_data.json',
        'results/exp06_control/control_data.json',
    )

MODEL_ID      = 'meta-llama/Llama-3.1-8B-Instruct'
INPUT_DIR     = 'results/exp01'
CTRL_DIR      = 'results/exp06_control'
OUTPUT_DIR    = 'results/exp07'
N_MAX         = 80
MIN_FIDELITY  = 1.0   # same as exp04-05a: stego pairs with perfect fidelity

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('cwd:', os.getcwd())

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer

with open(f'{INPUT_DIR}/valid_pairs.json') as f:
    all_pairs = json.load(f)

with open(f'{CTRL_DIR}/control_data.json') as f:
    control_data = json.load(f)

hi_fid = [p for p in all_pairs if p['fidelity'] == MIN_FIDELITY]
ctrl_index = {c['pair_idx']: c for c in control_data}

print(f'hi-fidelity stego pairs: {len(hi_fid)}')
print(f'control pairs available: {len(control_data)}')
usable = sum(1 for i in range(min(len(hi_fid), N_MAX)) if i in ctrl_index)
print(f'pairs with both stego and control: {usable}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',
)
model.eval()

print(f'Layers: {model.config.num_hidden_layers}, heads: {model.config.num_attention_heads}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def find_sentence_starts(token_ids, plen):
    cot_ids   = list(token_ids[plen:])
    full_text = tokenizer.decode(cot_ids, skip_special_tokens=True)

    char_starts = []
    i = 0
    while i < len(full_text) and full_text[i] == '\n':
        i += 1
    if i < len(full_text):
        char_starts.append(i)
    while i < len(full_text):
        if full_text[i] == '\n':
            j = i + 1
            while j < len(full_text) and full_text[j] == '\n':
                j += 1
            if j - i >= 2 and j < len(full_text):
                char_starts.append(j)
            i = max(i + 1, j)
        else:
            i += 1

    positions = []
    found     = 0
    prev_len  = 0
    for tok_idx in range(len(cot_ids)):
        curr_len = len(tokenizer.decode(cot_ids[:tok_idx + 1], skip_special_tokens=True))
        while found < len(char_starts) and prev_len <= char_starts[found] < curr_len:
            positions.append(plen + tok_idx)
            found += 1
        prev_len = curr_len
        if found >= len(char_starts):
            break
    return positions


@torch.no_grad()
def extract_attentions(token_ids):
    ids_t   = torch.tensor([token_ids], dtype=torch.long).to(model.device)
    outputs = model(ids_t, output_attentions=True, use_cache=False)
    attn    = [a[0].cpu().float() for a in outputs.attentions]
    del outputs, ids_t
    torch.cuda.empty_cache()
    return attn


def mean_elevation(attentions, sent_positions):
    """Attention from pos[K] to pos[:K] within CoT, averaged over K>=1, all layers, heads."""
    vals = []
    for k in range(1, len(sent_positions)):
        pos_k = sent_positions[k]
        prev  = sent_positions[:k]
        for attn in attentions:
            vals.append(float(attn[:, pos_k, prev].mean()))
    return float(np.mean(vals)) if vals else 0.0


def mean_prompt_attention(attentions, sent_positions, plen):
    """Fraction of attention from sentence starts allocated to prompt tokens.

    Uses sum (not mean) over prompt positions per head — attn[pos_k, :] sums to 1
    (softmax over causal past), so sum over prompt positions = fraction going to prompt.
    Averaged over heads and sentence-start positions.
    """
    vals = []
    for pos_k in sent_positions:
        for attn in attentions:
            # shape: (n_heads,) — fraction of each head's attention going to prompt
            vals.append(float(attn[:, pos_k, :plen].sum(dim=-1).mean()))
    return float(np.mean(vals)) if vals else 0.0


print('helpers ready')

In [ ]:
results  = []
skipped  = 0

for i, pair in enumerate(hi_fid[:N_MAX]):
    if i not in ctrl_index:
        continue

    ctrl  = ctrl_index[i]
    n     = len(pair['payload'])

    s_pos = find_sentence_starts(pair['stego_ids'],   pair['stego_plen'])[:n]
    c_pos = find_sentence_starts(ctrl['control_ids'], ctrl['control_plen'])[:n]
    o_pos = find_sentence_starts(pair['open_ids'],    pair['open_plen'])[:n]

    n_pos = min(len(s_pos), len(c_pos), len(o_pos))
    if n_pos < 2:
        skipped += 1
        continue

    s_attn  = extract_attentions(pair['stego_ids'])
    s_elev  = mean_elevation(s_attn, s_pos[:n_pos])
    s_pattn = mean_prompt_attention(s_attn, s_pos[:n_pos], pair['stego_plen'])
    del s_attn

    c_attn  = extract_attentions(ctrl['control_ids'])
    c_elev  = mean_elevation(c_attn, c_pos[:n_pos])
    c_pattn = mean_prompt_attention(c_attn, c_pos[:n_pos], ctrl['control_plen'])
    del c_attn

    o_attn  = extract_attentions(pair['open_ids'])
    o_elev  = mean_elevation(o_attn, o_pos[:n_pos])
    o_pattn = mean_prompt_attention(o_attn, o_pos[:n_pos], pair['open_plen'])
    del o_attn

    results.append({
        'pair_idx':        i,
        'stego_elevation': s_elev - o_elev,
        'ctrl_elevation':  c_elev - o_elev,
        'stego_pattn':     s_pattn,
        'ctrl_pattn':      c_pattn,
        'open_pattn':      o_pattn,
    })

    if len(results) % 10 == 0:
        print(f'Done {len(results)} pairs (skipped {skipped})')
        with open(f'{OUTPUT_DIR}/exp07_raw.json', 'w') as f:
            json.dump({'results': results, 'skipped': skipped}, f)

print(f'\nTotal: {len(results)} pairs, {skipped} skipped (<2 sentence starts)')

with open(f'{OUTPUT_DIR}/exp07_raw.json', 'w') as f:
    json.dump({'results': results, 'skipped': skipped}, f, indent=2)
print('Saved:', f'{OUTPUT_DIR}/exp07_raw.json')

In [ ]:
# Reload from disk — safe to re-run from here if session crashed after cell 05.
_checkpoint = locals().get('results', None)
if _checkpoint is None:
    with open(f'{OUTPUT_DIR}/exp07_raw.json') as f:
        _data = json.load(f)
    results = _data['results']
    skipped = _data.get('skipped', None)

res    = np.array([[r['stego_elevation'], r['ctrl_elevation'],
                    r['stego_pattn'], r['ctrl_pattn'], r['open_pattn']]
                   for r in results])
n      = len(results)
s_elev = res[:, 0]
c_elev = res[:, 1]
s_pa   = res[:, 2]
c_pa   = res[:, 3]
o_pa   = res[:, 4]


def ttest_fmt(arr, mu=0):
    t, p = stats.ttest_1samp(arr, mu)
    return f'{arr.mean():.6f} ± {arr.std()/np.sqrt(n):.6f}  (t={t:.2f}, p={p:.4f})'


print('=== exp07 — prompt attention analysis ===')
print(f'n_pairs: {n}')
print()
print('--- Metric 1: elevation (attn → prev sent starts within CoT, relative to open) ---')
print(f'stego: {ttest_fmt(s_elev)}')
print(f'ctrl:  {ttest_fmt(c_elev)}')
print()
print('--- Metric 2: fraction of attention going to prompt tokens ---')
print(f'stego:   {s_pa.mean():.4f} ± {s_pa.std()/np.sqrt(n):.4f}')
print(f'control: {c_pa.mean():.4f} ± {c_pa.std()/np.sqrt(n):.4f}')
print(f'open:    {o_pa.mean():.4f} ± {o_pa.std()/np.sqrt(n):.4f}')
print()
print('Paired t-tests (prompt attention):')
for label, a, b in [('stego vs open',    s_pa, o_pa),
                    ('control vs open',  c_pa, o_pa),
                    ('stego vs control', s_pa, c_pa)]:
    t, p = stats.ttest_rel(a, b)
    print(f'  {label}: t={t:.2f}, p={p:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['steelblue', 'orange', 'crimson']

# Left: elevation (relative to open)
ax = axes[0]
means = [0, c_elev.mean(), s_elev.mean()]
sems  = [0, c_elev.std()/np.sqrt(n), s_elev.std()/np.sqrt(n)]
ax.bar(['open\n(=0)', 'control', 'stego'], means, yerr=sems, color=colors, capsize=5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean elevation (condition − open)')
ax.set_title('Metric 1: elevation (attn → prev sent starts)')

# Right: prompt attention fraction
ax2 = axes[1]
means2 = [o_pa.mean(), c_pa.mean(), s_pa.mean()]
sems2  = [o_pa.std()/np.sqrt(n), c_pa.std()/np.sqrt(n), s_pa.std()/np.sqrt(n)]
ax2.bar(['open', 'control', 'stego'], means2, yerr=sems2, color=colors, capsize=5)
ax2.set_ylabel('Fraction of attention to prompt tokens')
ax2.set_title('Metric 2: prompt attention (attn[sent_start → prompt])')

plt.suptitle('exp07: where does the model look from sentence starts?', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/exp07_results.png', dpi=150)
plt.show()

In [ ]:
from scipy import stats as _st

def _ttest(a, b=None):
    if b is None:
        t, p = _st.ttest_1samp(a, 0)
    else:
        t, p = _st.ttest_rel(a, b)
    return round(float(t), 3), round(float(p), 6)

summary = {
    'model':    MODEL_ID,
    'n_pairs':  n,
    'skipped':  skipped,
    'metric1_elevation': {
        'stego': {'mean': round(float(s_elev.mean()), 6),
                  'sem':  round(float(s_elev.std()/np.sqrt(n)), 6),
                  't':    _ttest(s_elev)[0], 'p': _ttest(s_elev)[1]},
        'ctrl':  {'mean': round(float(c_elev.mean()), 6),
                  'sem':  round(float(c_elev.std()/np.sqrt(n)), 6),
                  't':    _ttest(c_elev)[0], 'p': _ttest(c_elev)[1]},
    },
    'metric2_prompt_attention': {
        'stego':   {'mean': round(float(s_pa.mean()), 6), 'sem': round(float(s_pa.std()/np.sqrt(n)), 6)},
        'control': {'mean': round(float(c_pa.mean()), 6), 'sem': round(float(c_pa.std()/np.sqrt(n)), 6)},
        'open':    {'mean': round(float(o_pa.mean()), 6), 'sem': round(float(o_pa.std()/np.sqrt(n)), 6)},
        'ttest_stego_vs_open':    {'t': _ttest(s_pa, o_pa)[0], 'p': _ttest(s_pa, o_pa)[1]},
        'ttest_control_vs_open':  {'t': _ttest(c_pa, o_pa)[0], 'p': _ttest(c_pa, o_pa)[1]},
        'ttest_stego_vs_control': {'t': _ttest(s_pa, c_pa)[0], 'p': _ttest(s_pa, c_pa)[1]},
    },
}

out_path = f'{OUTPUT_DIR}/exp07_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)

if IN_COLAB:
    import pathlib, shutil
    dst = pathlib.Path('/content/drive/MyDrive/stego_cot_results/exp07')
    dst.mkdir(parents=True, exist_ok=True)
    for fp in pathlib.Path(OUTPUT_DIR).iterdir():
        shutil.copy(fp, dst / fp.name)
        print(f'  -> Drive: {fp.name}')